# Etapa 6: Feature Engineering
---

In [2]:
# Imports
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
# Ruta raíz del proyecto (cwd = donde se encuentra el notebook; .parent = ruta padre, eso da la ruta raíz)
PROJECT_ROOT = Path.cwd().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bank_marketing.csv"

df = pd.read_csv(PROCESSED_PATH)

In [4]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

## Preprocessor (codificación, escalado)

In [ ]:
from src.features.feature_engineering import build_pipeline

preprocessor = build_pipeline(incluir_escalado=True)
print(preprocessor)

ColumnTransformer(transformers=[('onehot',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome']),
                                ('scaler', RobustScaler(),
                                 ['age', 'balance', 'day', 'campaign', 'pdays',
                                  'previous'])])


## Repaso del decisiones


### Codificación

Para codificar las categorías de cada una de las variables predictoras se usará One-Hot Encoding. La razón principal es porque, mientras los modelos de KNN y RL se basen en distancias, al usar otro método como Label Encoding, interpretarían que una categoría está más lejana que otra; con One-Hot Encoding evita esos errores al crear una columna booleana indicando a cual categoría pertenece la instancia, marcandola con un `1` y a las que no pertenece con `0`.

En cuanto a la conocida "maldición de la dimensionalidad" en machine learning, no es un problema en este caso, ya codificadas el total de columnas sería aproximadament 50. Contemplando que se cuenta con ~45,000 instancias, tener 50 columnas no será un problema en el entrenamiento.

La variable target `y` se excluye de dicha codificación, simplemente se codifica como 0 si `no` y 1 si `yes`.

### Escalado

El escalado aplica unicamente para el modelo de KNN y el de Regresión Logística. Esto debido a que ambos modelos requieren escalado a diferencia de los árboles. Por un lado porque KNN funciona por distancias, por ende, si se mantienen rangos muy distintos como es el caso de `balance` y `age`, `age` quedará casi completamente dominado por la diferencia de `balance`; mientras que Regresión Logistica lo requiere porque se entrena con regularización y esta va a penalizar de forma desigual a las escalas grandes como en el caso de `balance`.

Se considera ideal usar el escalado RobustScaler. La decisión se justifica en base a que los otros métodos más conocidos son bastante sensibles a outliers, y durante el diagnóstico se identificaron casos de outliers y de asimetría en algunas de las variables numéricas que estas se consideraron casos posibles del negocio.

---

In [6]:
# Separación de las variables predictoras (X) de la variable objetivo (y)
# Además, se elimina 'duration' porque podría introducir data leakage en el modelo
X = df.drop(columns=["y", "duration"])
y = df["y"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# Verificar las variables que serán utilizadas como predictoras
print("Variables predictoras:")
print(X.columns.tolist())

Dimensiones de X: (45211, 15)
Dimensiones de y: (45211,)
Variables predictoras:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [7]:
from sklearn.model_selection import train_test_split

# Dividimos los datos en entrenamiento (80%) y prueba (20%).
# stratify mantiene aproximadamente la misma proporción de clases yes/no en ambos conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
# Comparamos la distribución de la variable objetivo entre entrenamiento y prueba
# para comprobar que el desbalance de clases se mantiene aproximadamente igual.

print("Distribución en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True).round(3))

Distribución en entrenamiento:
y
no     0.883
yes    0.117
Name: proportion, dtype: float64

Distribución en prueba:
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


In [ ]:
# Creamos el pipeline de Decision Tree.
# Primero transforma las variables y después aplica el modelo.
decision_tree_pipeline = build_pipeline(
    model=DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=4, criterion="gini"),
    incluir_escalado=False
)

print("Pipeline de Decision Tree creado correctamente.")

Pipeline de Decision Tree creado correctamente.


In [ ]:
# Creamos el pipeline de Random Forest.
# Primero transforma las variables y después aplica el modelo.
random_forest_pipeline = build_pipeline(
    model=RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=100, max_depth=5, n_jobs=-1),
    incluir_escalado=False
)

print("Pipeline de Random Forest creado correctamente.")

Pipeline de Random Forest creado correctamente.


In [ ]:
k=2 # TODO

# Creamos el pipeline de KNN
# Primero transforma las variables, aplica escalado y después corre el modelo.
knn_pipeline = build_pipeline(
    model=KNeighborsClassifier(n_neighbors=k),
    incluir_escalado=True,
    incluir_smote=True   # KNN no soporta class_weight
)

print("Pipeline de KNN creado correctamente.")

Pipeline de KNN creado correctamente.


In [ ]:
# Creamos el pipeline de Regresión Logística
# Primero transforma las variables, aplica escalado y después corre el modelo.
rl_pipeline = build_pipeline(
    model=LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    incluir_escalado=True
)

print("Pipeline de Regresión Logística creado correctamente.")

Pipeline de Regresión Logística creado correctamente.
